# Theory of Mind (ToM) Steering Vectors for Gemma-3-4B

This notebook trains steering vectors to enhance Theory of Mind capabilities in Gemma-3-4B using the repeng library.

**What this does:**
- Trains control vectors that steer the model toward better understanding of mental states, beliefs, and intentions
- Exports vectors in `.gguf` format for later use
- Provides both a general ToM vector and specialized vectors for specific ToM skills

**Repository:** https://github.com/ChuloIva/Cogni_map

## 1. Setup & Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the Cogni_map repository
!git clone https://github.com/ChuloIva/Cogni_map.git
%cd Cogni_map

In [ ]:
# Install dependencies
!pip install -q transformers torch accelerate sentencepiece

# Install the repeng library from the cloned repo
!pip install -q -e ToM/repeng/

In [ ]:
# Optional: Mount Google Drive to save vectors permanently
from google.colab import drive
drive.mount('/content/drive')

# Create directory for saving vectors
!mkdir -p /content/drive/MyDrive/tom_steering_vectors

## 2. Load Model (Gemma-3-4B, Text-Only)

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoConfig, Gemma3ForCausalLM, AutoTokenizer
from repeng import ControlVector, ControlModel, DatasetEntry

In [ ]:
# Model configuration
model_name = "google/gemma-3-4b-it"

print(f"Loading {model_name}...")

# Load config to check if it's a vision-language model
config = AutoConfig.from_pretrained(model_name)

# Load model (skip vision tower if present)
if hasattr(config, 'vision_config'):
    print("Detected vision-language model. Loading text-only version (skipping vision tower)...")
    base_model = Gemma3ForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    print("Loading standard causal LM...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = 0  # Set padding token

print("Model loaded successfully!")
print(f"Device: {base_model.device}")

In [ ]:
# Wrap with ControlModel for steering
# Using layers -5 to -18 (last 13 layers, counting from the end)
model = ControlModel(base_model, list(range(-5, -18, -1)))

print(f"ControlModel initialized with layers: {list(range(-5, -18, -1))}")
print("Model ready for training steering vectors!")

## 3. Load Training Data

In [ ]:
# Load truncated output suffixes (conversation continuations)
with open("data/datagen/all_truncated_outputs.json") as f:
    output_suffixes = json.load(f)

print(f"Loaded {len(output_suffixes)} output suffixes for training")
print(f"Examples: {output_suffixes[:5]}")

In [ ]:
# Helper function to create datasets
def make_dataset(template: str, pos_personas: list[str], neg_personas: list[str], suffixes: list[str]):
    """
    Create a dataset of positive/negative pairs for training control vectors.
    
    Args:
        template: String template with {persona} placeholder
        pos_personas: List of positive persona descriptions
        neg_personas: List of negative persona descriptions (same length as pos_personas)
        suffixes: List of text suffixes to append to each persona
    
    Returns:
        List of DatasetEntry objects
    """
    dataset = []
    for suffix in suffixes:
        for positive_persona, negative_persona in zip(pos_personas, neg_personas):
            dataset.append(
                DatasetEntry(
                    positive=template.format(persona=positive_persona) + suffix,
                    negative=template.format(persona=negative_persona) + suffix,
                )
            )
    return dataset

print("Dataset creation helper function defined.")

## 4. Train Primary ToM Vector (BEST ONE)

This is the recommended general-purpose ToM steering vector from the documentation.

In [ ]:
# Create the general ToM dataset
# This contrasts good vs bad theory of mind capabilities
tom_dataset = make_dataset(
    "You're {persona}.",
    ["excellent at understanding minds, predicting behavior, and inferring what others think and feel"],
    ["terrible at understanding minds, predicting behavior, and inferring what others think and feel"],
    output_suffixes
)

print(f"Created ToM dataset with {len(tom_dataset)} training pairs")
print(f"\nExample pair:")
print(f"  Positive: {tom_dataset[0].positive[:120]}...")
print(f"  Negative: {tom_dataset[0].negative[:120]}...")

In [ ]:
# Train the general ToM vector
print("Training general ToM steering vector...")
print("This may take a few minutes...\n")

model.reset()  # Always reset before training
tom_vector = ControlVector.train(model, tokenizer, tom_dataset)

print("\nTraining complete!")
print(f"Vector contains directions for {len(tom_vector.directions)} layers")

In [ ]:
# Export the vector
vector_path = "tom_general.gguf"
tom_vector.export_gguf(vector_path)
print(f"Exported to: {vector_path}")

# Also save to Google Drive (if mounted)
try:
    import shutil
    drive_path = f"/content/drive/MyDrive/tom_steering_vectors/{vector_path}"
    shutil.copy(vector_path, drive_path)
    print(f"Also saved to Google Drive: {drive_path}")
except Exception as e:
    print(f"Could not save to Google Drive: {e}")

## 5. Optional: Train Specialized ToM Vectors

These vectors target specific Theory of Mind skills. Uncomment any section to train that vector.

### 5.1 Forward Belief Vector
Tracks what someone believes based on what they've seen/haven't seen (false belief reasoning).

In [ ]:
# # Uncomment to train forward_belief vector
# forward_belief_dataset = make_dataset(
#     "You're {persona} at figuring out what people believe given what they've seen or haven't seen.",
#     [
#         "accurate at maintaining false beliefs when someone didn't see a change",
#         "precise at understanding people maintain outdated beliefs when unaware",
#         "skilled at tracking what someone believes vs what's actually true"
#     ],
#     [
#         "always confused between what people believe and what's actually true",
#         "incorrectly updating beliefs even when someone didn't witness changes",
#         "unable to distinguish between what someone knows vs doesn't know"
#     ],
#     output_suffixes
# )
# 
# print(f"Training forward_belief vector ({len(forward_belief_dataset)} pairs)...")
# model.reset()
# forward_belief_vector = ControlVector.train(model, tokenizer, forward_belief_dataset)
# forward_belief_vector.export_gguf("tom_forward_belief.gguf")
# print("Exported to: tom_forward_belief.gguf")

### 5.2 Forward Action Vector
Predicts what someone will do based on their beliefs and desires.

In [ ]:
# # Uncomment to train forward_action vector
# forward_action_dataset = make_dataset(
#     "You {persona} predict what people will do based on their beliefs and what they want.",
#     [
#         "correctly predict actions that follow from desires and beliefs",
#         "understand that people act on beliefs, not necessarily reality",
#         "accurately infer actions by connecting desires → beliefs → actions"
#     ],
#     [
#         "predict actions based only on reality, ignoring beliefs",
#         "can't connect desires and beliefs to predict behaviors",
#         "think people always act on complete accurate information"
#     ],
#     output_suffixes
# )
# 
# print(f"Training forward_action vector ({len(forward_action_dataset)} pairs)...")
# model.reset()
# forward_action_vector = ControlVector.train(model, tokenizer, forward_action_dataset)
# forward_action_vector.export_gguf("tom_forward_action.gguf")
# print("Exported to: tom_forward_action.gguf")

### 5.3 Backward Belief Vector
Infers what someone believes from their observed actions.

In [ ]:
# # Uncomment to train backward_belief vector
# backward_belief_dataset = make_dataset(
#     "When you see what someone does, you {persona} infer what they believe.",
#     [
#         "correctly infer beliefs from observed actions",
#         "accurately reason backward from behaviors to mental states",
#         "understand actions reveal beliefs"
#     ],
#     [
#         "can't reason backward from actions to beliefs",
#         "ignore actions when inferring what people think",
#         "don't realize actions provide clues about beliefs"
#     ],
#     output_suffixes
# )
# 
# print(f"Training backward_belief vector ({len(backward_belief_dataset)} pairs)...")
# model.reset()
# backward_belief_vector = ControlVector.train(model, tokenizer, backward_belief_dataset)
# backward_belief_vector.export_gguf("tom_backward_belief.gguf")
# print("Exported to: tom_backward_belief.gguf")

### 5.4 Percept-to-Belief Vector
Maps what someone perceives to what they come to believe.

In [ ]:
# # Uncomment to train percept_to_belief vector
# percept_to_belief_dataset = make_dataset(
#     "You {persona} connect what people perceive to what they come to believe.",
#     [
#         "accurately map perceptions to beliefs",
#         "understand that perception leads to belief formation",
#         "correctly reason: perceive → believe"
#     ],
#     [
#         "fail to connect perception to belief",
#         "can't map what people see to what they think",
#         "don't understand perception cues indicate beliefs"
#     ],
#     output_suffixes
# )
# 
# print(f"Training percept_to_belief vector ({len(percept_to_belief_dataset)} pairs)...")
# model.reset()
# percept_to_belief_vector = ControlVector.train(model, tokenizer, percept_to_belief_dataset)
# percept_to_belief_vector.export_gguf("tom_percept_to_belief.gguf")
# print("Exported to: tom_percept_to_belief.gguf")

### 5.5 Combined Specialized Vector
Uncomment to create a vector that combines all four specialized skills.

In [ ]:
# # Uncomment to train all 4 specialized vectors and combine them
# # (Make sure you've uncommented and run all 4 sections above first!)
# 
# combined_tom_vector = (
#     forward_belief_vector +
#     forward_action_vector + 
#     backward_belief_vector +
#     percept_to_belief_vector
# ) / 4  # Average the vectors
# 
# combined_tom_vector.export_gguf("tom_combined_specialized.gguf")
# print("Exported combined vector to: tom_combined_specialized.gguf")

## 6. Test the ToM Vector

In [ ]:
# Helper function for generation
def generate_text(prompt, model, tokenizer, max_new_tokens=128):
    """
    Generate text from the model.
    """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Deterministic
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1
    )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Test prompt: A classic false belief scenario
test_prompt = """Sarah puts her toy in the red box and leaves the room. 
While she's gone, John moves the toy to the blue box. 
When Sarah returns, where will she look for her toy?

Answer:"""

print("Testing ToM steering vector...\n")
print("="*80)

# Baseline (no steering)
print("\n[BASELINE - No Steering]")
model.reset()
baseline_output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(baseline_output)

# With positive ToM steering
print("\n" + "="*80)
print("\n[WITH ToM STEERING - Strength: 1.5]")
model.set_control(tom_vector, coeff=1.5)
steered_output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(steered_output)

# With negative ToM steering (anti-ToM)
print("\n" + "="*80)
print("\n[ANTI-ToM STEERING - Strength: -2.0]")
model.set_control(tom_vector, coeff=-2.0)
anti_steered_output = generate_text(test_prompt, model, tokenizer, max_new_tokens=50)
print(anti_steered_output)

# Reset model
model.reset()
print("\n" + "="*80)

### Test with Your Own Prompts

In [ ]:
# Try your own ToM scenario
custom_prompt = """John thinks the meeting is at 3pm, but it was changed to 2pm. 
He wasn't informed about the change. What time will John show up?

Answer:"""

print("Custom test:")
print("="*80)
model.set_control(tom_vector, coeff=1.5)
result = generate_text(custom_prompt, model, tokenizer, max_new_tokens=100)
print(result)
model.reset()

## 7. Download Vectors

Download the trained vectors to your local machine.

In [ ]:
from google.colab import files
import os

# List all .gguf files in current directory
gguf_files = [f for f in os.listdir('.') if f.endswith('.gguf')]

print(f"Found {len(gguf_files)} vector file(s):")
for f in gguf_files:
    print(f"  - {f}")

# Download each file
print("\nDownloading...")
for f in gguf_files:
    files.download(f)
    print(f"Downloaded: {f}")

print("\nAll vectors downloaded!")

## 8. How to Use the Vectors Later

To use these vectors in future sessions:

In [ ]:
# # Example: Load and use a saved vector
# from repeng import ControlVector, ControlModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
# 
# # Load model
# model = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it")
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
# model = ControlModel(model, list(range(-5, -18, -1)))
# 
# # Load the vector
# tom_vector = ControlVector.import_gguf("tom_general.gguf")
# 
# # Apply the vector
# model.set_control(tom_vector, coeff=1.5)
# 
# # Generate with ToM enhancement
# # ... your generation code ...
# 
# # Reset when done
# model.reset()

## Notes

**Vector Strength (coeff parameter):**
- Start with values between -2.5 and 2.5
- Positive values: Enhance ToM capabilities
- Negative values: Reduce ToM capabilities (useful for testing)
- Typical good range: 1.0 to 2.0

**Best Practices:**
- Always call `model.reset()` before training a new vector
- Always call `model.reset()` after generation if you want baseline behavior
- Export vectors immediately after training to avoid losing them
- Test different coefficient values to find what works best for your use case

**Memory Tips:**
- If you run out of memory, restart runtime and reduce batch size in training
- Use `torch.float16` (already configured) to save memory
- Consider using fewer layers in ControlModel if needed